# Deeptech M&A Momentum: Feature Engineering

## Phase 3, Step 3.2: Transformation for Predictive Modeling

This notebook performs the final **feature engineering** required to prepare the M&A volume and ETF return time series for prediction. Instead of performing iterative statistical checks, we apply the following predetermined transformations to construct robust input features:

1.  **M&A Volume (Predictor):** The raw volume is transformed using **first-order differencing** ($\Delta V_t$) to enforce stationarity and then **Min-Max Scaled** ($V^{norm}_t$) to normalize the feature magnitude for modeling. 
2.  **Sector Returns (Target):** Simple monthly returns ($R_t$) are retained, assuming stationarity.

**We will iterate through all three look-back frequencies ('1mo', '3mo', '6mo') to create three distinct sets of aligned and engineered features for subsequent causality testing and modeling.**

---

### Setup and configuration

In [1]:
# --- 1. Standard Library Imports ---
from pathlib import Path
import sys
from typing import Dict, List

# --- 2. Third-Party Library Imports ---\n",
import polars as pl
import numpy as np 
import pandas as pd

# --- 3. Local Application Imports ---
sys.path.append(str(Path.cwd().parent.parent / "src"))
import market_data

In [2]:
# --- Configuration ---
TEST_FREQUENCIES = ["1mo", "3mo", "6mo"]
MARKET_INTERVAL = '1mo' 
START_DATE = "2018-01-01"
END_DATE = "2024-01-01"
# MAX_DIFF_ORDER is no longer relevant

# --- SECTOR-TICKER MAP ---\n",
SECTOR_TICKER_MAP: Dict[str, str] = {
    "Green Hydrogen Infrastructure": "HYDR",
    "Communications and Networks, including 5G": "IYZ",
    "Energy Transition Metals": "TMET",
    "Robotics": "BOTZ",
    "Aerospace Defense Systems Integration": "ITA",
    "Internet of Things, W3C, Semantic Web": "SNSR",
    "Artificial Intelligence and Machine Learning, including Big Data": "AIQ",
    "Semiconductors (microchips)": "SOXX",
    "Advanced Battery Chemistry / Storage": "LIT",
    "Additive Manufacturing / 3D Printing": "PRNT",
    "Solar Grid Optimization / Smart Grid": "GRID",
    "Gene Therapy / CRISPR": "ARKG",
    "Carbon Capture & Storage (CCS) / Geoengineering": "KRBN",
    "High-Performance Composites & Ceramics": "XLB",
    "Functional Surfaces & Coatings": "XLB", 
    "Precision Machining and Metrology": "IYJ",
    "Offshore Wind Farm Development": "FAN"
}

MNA_INPUT_DIR = Path("../../data/processed")

# Define the three frequencies we need to merge
VOLUME_FREQUENCIES = ["1mo", "3mo", "6mo"]

# Helper function for Z-Score Standardization
def z_score_standardize(series):
    """Applies Z-Score Standardization: (x - mean) / std_dev. Handles zero variance."""
    s_mean = series.mean().over("Ticker")
    s_std = series.std().over("Ticker")
    
    # CRITICAL FIX: If standard deviation is zero (zero variance data), 
    # the formula becomes division by zero. We set the result to 0.0 (or a small constant).
    return pl.when(s_std == 0.0).then(0.0).otherwise((series - s_mean) / s_std)

print(f"Market Interval for Returns: {MARKET_INTERVAL}")

Market Interval for Returns: 1mo


### Step 1: Data Acquisition (Market Returns)

In [3]:
# --- Step 1: Data Acquisition (Market Returns) ---
sector_list = list(SECTOR_TICKER_MAP.values())
all_tickers = list(set(sector_list + [market_data.BENCHMARK_TICKER])) 

df_prices = market_data.get_historical_prices(
    all_tickers,
    START_DATE,
    END_DATE,
    interval=MARKET_INTERVAL
)

# Calculate multi-horizon compound returns
df_returns = df_prices.sort("Ticker", "Date").with_columns([
    # 1-month return: P_t / P_{t-1} - 1
    (pl.col("Adj_Close") / pl.col("Adj_Close").shift(1).over("Ticker") - 1).alias("Returns_Target_1m"),
    
    # 3-month compound return: P_t / P_{t-3} - 1
    (pl.col("Adj_Close") / pl.col("Adj_Close").shift(3).over("Ticker") - 1).alias("Returns_Target_3m"),
    
    # 6-month compound return: P_t / P_{t-6} - 1
    (pl.col("Adj_Close") / pl.col("Adj_Close").shift(6).over("Ticker") - 1).alias("Returns_Target_6m")
])

# Drop rows where the 6-month return could not be calculated (first 6 months of data) and drop Adj_Close
df_returns = df_returns.drop_nulls(subset=["Returns_Target_6m"]).drop("Adj_Close")

print(f"✓ Calculated multi-horizon returns for {df_returns.get_column('Ticker').n_unique()} tickers.")

Fetching 1mo data for 17 tickers...



1 Failed download:
['TMET']: YFPricesMissingError('possibly delisted; no price data found  (1mo 2018-01-01 -> 2024-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1514782800, endDate = 1704085200")')


✓ Fetched 1,076 market data points.

Effective Start Dates per Ticker (May differ from requested start_date):
| Ticker   | Effective_Start_Date   |
|:---------|:-----------------------|
| SNSR     | 2018-01-01 00:00:00    |
| PRNT     | 2018-01-01 00:00:00    |
| SOXX     | 2018-01-01 00:00:00    |
| XLB      | 2018-01-01 00:00:00    |
| ITA      | 2018-01-01 00:00:00    |
| ARKG     | 2018-01-01 00:00:00    |
| IYJ      | 2018-01-01 00:00:00    |
| ^GSPC    | 2018-01-01 00:00:00    |
| GRID     | 2018-01-01 00:00:00    |
| BOTZ     | 2018-01-01 00:00:00    |
| IYZ      | 2018-01-01 00:00:00    |
| LIT      | 2018-01-01 00:00:00    |
| FAN      | 2018-01-01 00:00:00    |
| AIQ      | 2018-05-01 00:00:00    |
| KRBN     | 2020-07-01 00:00:00    |
| HYDR     | 2021-07-01 00:00:00    |
✓ Calculated multi-horizon returns for 16 tickers.


### Step 2: M\&A Volume Feature Engineering (Differencing and Scaling)

In [4]:
# --- Step 2: M&A Volume Feature Engineering (Multi-Frequency Merge) ---

# Define the columns that represent the sector time series in the wide CSV
MNA_SECTOR_COLUMNS = list(SECTOR_TICKER_MAP.keys())

# Start with the df_returns base (which contains all three return targets)
df_combined = df_returns

# Loop through all volume frequencies and merge them sequentially
for i, freq in enumerate(VOLUME_FREQUENCIES):
    print(f"\nProcessing M&A Volume Frequency: {freq}")
    
    INPUT_PATH = MNA_INPUT_DIR / f"3.0_sector_volume_{freq}.csv" 
    
    if not INPUT_PATH.exists():
        print(f"ERROR: Volume CSV file not found for {freq}. Skipping.")
        continue
        
    df_volume_wide = pl.read_csv(INPUT_PATH, try_parse_dates=True)
    df_volume_wide = df_volume_wide.rename({df_volume_wide.columns[0]: "Date"})
    
    # 1. Melt the wide data frame into a long format
    df_volume = df_volume_wide.unpivot(
        index=["Date"],
        on=MNA_SECTOR_COLUMNS, 
        variable_name="deeptech_sector",
        value_name="Volume_MNA_Raw"
    )

    # 2. Map Sector Name to Ticker
    df_volume = df_volume.with_columns(
        pl.col("deeptech_sector").replace(SECTOR_TICKER_MAP).alias("Ticker")
    ).drop("deeptech_sector")
    
    # 3. CRITICAL FIX: Aggregate volume for Tickers that map to multiple sectors (e.g., XLB)
    df_volume_agg = df_volume.group_by(["Date", "Ticker"]).agg(
        pl.col("Volume_MNA_Raw").sum().alias("Volume_MNA_Raw_Agg")
    )
    
    # Rename the aggregated raw volume column to be frequency-specific before joining
    raw_col_name = f"Volume_MNA_Raw_{freq}"
    df_volume_agg = df_volume_agg.rename({"Volume_MNA_Raw_Agg": raw_col_name})
    
    # 4. Join the aggregated volume stream (unique keys) to the combined dataset
    df_combined = df_combined.join(
        df_volume_agg.drop_nulls(),
        on=["Date", "Ticker"], 
        how="left"
    ).sort("Date")

# Get a list of all raw volume column names after merging
raw_volume_cols = [f"Volume_MNA_Raw_{f}" for f in VOLUME_FREQUENCIES]

# Fill missing volume values (where no deals occurred) with 0.0
df_combined = df_combined.with_columns(
    [pl.col(c).fill_null(0.0) for c in raw_volume_cols]
)

print(f"\n✓ All raw volume features merged into one DataFrame with aggregation.")


Processing M&A Volume Frequency: 1mo

Processing M&A Volume Frequency: 3mo

Processing M&A Volume Frequency: 6mo

✓ All raw volume features merged into one DataFrame with aggregation.


In [5]:
# --- Step 3: Transformation and Final Save ---\n",

print("\n" + "=" * 80)
print("PHASE 3.2 | Applying Transformations to All Frequencies")
print("=" * 80)

# 1. Apply Differencing (for all three raw volume columns)
diff_expressions = [
    pl.col(c).diff().over("Ticker").alias(c.replace("Raw", "Diff"))
    for c in raw_volume_cols
]
df_processed = df_combined.with_columns(diff_expressions)

# 2. Apply Z-Score Standardization on the Differenced Volume
# This is done PER-SECTOR (over("Ticker"))
scaled_expressions = [
    z_score_standardize(pl.col(c.replace("Raw", "Diff"))).alias(c.replace("Raw", "Scaled"))
    for c in raw_volume_cols
]
df_processed = df_processed.with_columns(scaled_expressions)

# 3. Final Selection and Save 
target_cols = ["Returns_Target_1m", "Returns_Target_3m", "Returns_Target_6m"]
scaled_volume_cols = [c.replace("Raw", "Scaled") for c in raw_volume_cols]

final_cols = ["Date", "Ticker"] + target_cols + scaled_volume_cols

df_final = df_processed.select(final_cols).drop_nulls()

# Save the master file containing all features
OUTPUT_PATH = MNA_INPUT_DIR / "3.2_aligned_features_MASTER.csv"
df_final.write_csv(OUTPUT_PATH)

print("\n" + "-" * 40)
print(f"✓ MASTER Feature file created successfully at {OUTPUT_PATH.name}")
print(f"Final rows: {len(df_final):,}\nFinal Feature Columns: {df_final.columns}")
print("-" * 44)


PHASE 3.2 | Applying Transformations to All Frequencies

----------------------------------------
✓ MASTER Feature file created successfully at 3.2_aligned_features_MASTER.csv
Final rows: 965
Final Feature Columns: ['Date', 'Ticker', 'Returns_Target_1m', 'Returns_Target_3m', 'Returns_Target_6m', 'Volume_MNA_Scaled_1mo', 'Volume_MNA_Scaled_3mo', 'Volume_MNA_Scaled_6mo']
--------------------------------------------
